In [13]:
import json
import pandas as pd
import yfinance as yf
from dateutil import parser as dateparser
import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import grangercausalitytests
from statsmodels.tsa.api import VAR

import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
START = "2010-01-01"
END = "2025-09-7"
BASE = "/home/influx/Desktop/FinLLMRL"
STOCKS = ['AAPL', 'BA', 'GS', 'JPM']

WINDOW = 7
HALF_LIFE = 2.5
MX_LAG = 10

In [15]:
def parse_date(item):
    """Parse ISO-like date string in item['date'] into pandas.Timestamp (date only)."""
    return pd.Timestamp(dateparser.parse(str(item['date'])).date())

In [16]:
# ---------- Helper function ----------
def signed_score(scores):
    if scores is None:
        return None
    return 0*scores['neutral'] + 1*scores['positive'] + (-1)*scores['negative']

In [17]:
def preparing_news(stock):
    with open(BASE + f"/Final_Data/{stock}_merged.json", "r") as f:
        news = json.load(f)

    data = {}
    for it in news:
        dt = parse_date(it)
        fb  = it.get("finbert_score", None)
        fg  = it.get("fingpt_score", None)
    
        if dt not in data:
            data[dt] = []
    
        data[dt].append({
            "finbert_score": signed_score(fb),
            "fingpt_score": signed_score(fg)
        })

    result = []
    
    for dt, items in data.items():
        finbert_scores = [it["finbert_score"] for it in items if it["finbert_score"] is not None]
        fingpt_scores  = [it["fingpt_score"]  for it in items if it["fingpt_score"]  is not None]
    
        avg_finbert = np.mean(finbert_scores) if finbert_scores else None
        avg_fingpt  = np.mean(fingpt_scores)  if fingpt_scores else None
    
        result.append({
            "date": dt,
            "avg_finbert": avg_finbert,
            "avg_fingpt": avg_fingpt
        })

    df = pd.DataFrame(result).sort_values("date")
    df = df.loc[
        (df['date'] >= pd.to_datetime(START)) &
        (df['date'] <= pd.to_datetime(END))
    ].reset_index(drop=True)

    return df

In [18]:
def preparing_OLCHV(stock):
    ST = pd.to_datetime(START)
    ED   = pd.to_datetime(END)
    
    prices = yf.download(
        stock,
        start=ST.strftime("%Y-%m-%d"),
        end=(ED + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        progress=False,
        auto_adjust=True
    ).reset_index()

    prices.columns = ["_".join([str(c) for c in col if c]).strip() for col in prices.columns.values]

    prices = prices.rename(columns={
        f"Close_{stock}": "Close",
        f"Volume_{stock}": "Volume",
        "Date": "date"
    })

    prices = prices.drop(columns=[f'High_{stock}', f'Low_{stock}', f'Open_{stock}'])
    return prices

In [19]:
def merge_data(prices, df):
    prices = prices.copy()
    df = df.copy()
    
    prices['date'] = pd.to_datetime(prices['date'])
    df['date'] = pd.to_datetime(df['date'])
    
    prices = prices.sort_values('date').drop_duplicates('date', keep='last')
    df = df.sort_values('date').drop_duplicates('date', keep='last')
    
    start = min(prices['date'].min(), df['date'].min())
    end   = max(prices['date'].max(), df['date'].max())
    all_days = pd.date_range(start, end, freq='D', name='date')  
    
    prices_full = (
        prices.set_index('date')
              .reindex(all_days)      
              .ffill()                
    )
    
    df_full = (
        df.set_index('date')
          .reindex(all_days, fill_value=0)
    )
    
    out = prices_full.join(df_full, how='outer').reset_index().rename(columns={'index':'date'})
    out = out.sort_values("date").reset_index(drop=True)
    out["return"] = np.log(out["Close"] / out["Close"].shift(1))
    out = out.dropna(subset=["return"]).reset_index(drop=True)

    out = out.sort_values('date').reset_index(drop=True)
    
    out['finbert7'] = past_exp_weighted_avg(out['avg_finbert'], window=WINDOW, half_life=HALF_LIFE, exclude_today=False)
    out['fingpt7']  = past_exp_weighted_avg(out['avg_fingpt'],  window=WINDOW, half_life=HALF_LIFE, exclude_today=False)

    return out

In [20]:
def past_exp_weighted_avg(series: pd.Series, window=7, half_life=1.5, exclude_today=False):
    """
    Exponentially weighted average over the past `window` days.
    - exclude_today=True -> uses t-1..t-window
    - Row-wise renormalization handles missing values at the start.
    """
    lags = np.arange(1, window + 1) if exclude_today else np.arange(0, window)
    lam = np.log(2) / half_life
    w = np.exp(-lam * lags)  

    lagged = pd.concat([series.shift(l) for l in lags], axis=1)

    weighted = lagged.mul(w, axis=1)
    weight_sums = (~lagged.isna()).mul(w, axis=1).sum(axis=1)
    out = weighted.sum(axis=1) / weight_sums
    out[weight_sums == 0] = np.nan
    return out

In [21]:
def adf_test(series, title=""):
    print(f"--- ADF Test: {title} ---")
    result = adfuller(series.dropna(), autolag='AIC')
    labels = ['ADF Statistic', 'p-value', '# Lags Used', '# Observations Used']
    out = dict(zip(labels, result[0:4]))
    for key, val in out.items():
        print(f"{key} : {val}")
    for key, val in result[4].items():
        print(f"Critical Value ({key}) : {val}")
    if result[1] <= 0.05:
        print("✅ Reject H0 → Stationary")
    else:
        print("❌ Fail to Reject H0 → Non-stationary")
    print("\n")

In [22]:
def granger_test(df, xvar, yvar="return", max_lag=5):
    print(f"\n=== Granger causality test: Does {xvar} → {yvar}? ===")
    test_result = grangercausalitytests(df[[yvar, xvar]], max_lag, verbose=True)
    return test_result

In [23]:
def granger_var_analysis(df, max_lag=10, target="return", predictors=["finbert7", "fingpt7"]):
    # Keep only relevant columns & drop missing
    cols = [target] + predictors
    data = df[cols].dropna()

    # Select optimal lag
    sel = VAR(data).select_order(max_lag)
    lag_candidates = [sel.aic, sel.bic, sel.hqic, sel.fpe]
    lag = next((int(x) for x in lag_candidates if x is not None and x > 0), 5)

    # Fit VAR
    var_res = VAR(data).fit(lag)

    print("="*50)
    print(f" Selected VAR Lag Order: {lag}")
    print("="*50)

    # Individual Granger causality tests
    results = []
    for pred in predictors:
        test = var_res.test_causality(target, [pred], kind="f")
        results.append((pred, test.test_statistic, test.pvalue))

    # Joint test
    test_joint = var_res.test_causality(target, predictors, kind="f")
    results.append(("Joint (" + ", ".join(predictors) + ")", 
                    test_joint.test_statistic, 
                    test_joint.pvalue))

    # Pretty print results
    print(f"\nGranger Causality Tests ({', '.join(predictors)} → {target})")
    print("-"*50)
    print(f"{'Predictor':<25}{'F-stat':>10}{'p-value':>12}")
    print("-"*50)
    for pred, fstat, pval in results:
        print(f"{pred:<25}{fstat:>10.3f}{pval:>12.4g}")
    print("-"*50)

    return results, var_res

# AAPL

In [24]:
prices = preparing_OLCHV("AAPL")
news = preparing_news("AAPL")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : 0.7348459716069242
p-value : 0.9905082324489315
# Lags Used : 33
# Observations Used : 5691
Critical Value (1%) : -3.4314995786368088
Critical Value (5%) : -2.86204800302548
Critical Value (10%) : -2.567040408291401
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -16.601806577934244
p-value : 1.761815398399324e-29
# Lags Used : 19
# Observations Used : 5705
Critical Value (1%) : -3.431496756314084
Critical Value (5%) : -2.862046756071205
Critical Value (10%) : -2.5670397445004576
✅ Reject H0 → Stationary




In [25]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [26]:
results, var_res = granger_var_analysis(df)

 Selected VAR Lag Order: 8

Granger Causality Tests (finbert7, fingpt7 → return)
--------------------------------------------------
Predictor                    F-stat     p-value
--------------------------------------------------
finbert7                      1.387      0.1966
fingpt7                       2.259     0.02072
Joint (finbert7, fingpt7)     2.812   0.0001412
--------------------------------------------------


# BA

In [27]:
prices = preparing_OLCHV("BA")
news = preparing_news("BA")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : -1.8183457544319275
p-value : 0.3713923222901341
# Lags Used : 34
# Observations Used : 5690
Critical Value (1%) : -3.4314997807629735
Critical Value (5%) : -2.862048092328505
Critical Value (10%) : -2.5670404558300723
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -12.967157465001595
p-value : 3.1359806153977597e-24
# Lags Used : 34
# Observations Used : 5690
Critical Value (1%) : -3.4314997807629735
Critical Value (5%) : -2.862048092328505
Critical Value (10%) : -2.5670404558300723
✅ Reject H0 → Stationary




In [28]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [29]:
results, var_res = granger_var_analysis(df)

 Selected VAR Lag Order: 10

Granger Causality Tests (finbert7, fingpt7 → return)
--------------------------------------------------
Predictor                    F-stat     p-value
--------------------------------------------------
finbert7                      0.865      0.5654
fingpt7                       1.791     0.05662
Joint (finbert7, fingpt7)     1.107      0.3329
--------------------------------------------------


# GS

In [30]:
prices = preparing_OLCHV("GS")
news = preparing_news("GS")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : 1.8951546517199451
p-value : 0.9985201835230484
# Lags Used : 31
# Observations Used : 5693
Critical Value (1%) : -3.431499174597602
Critical Value (5%) : -2.862047824513573
Critical Value (10%) : -2.5670403132641777
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -24.67455149134518
p-value : 0.0
# Lags Used : 8
# Observations Used : 5716
Critical Value (1%) : -3.4314945484779873
Critical Value (5%) : -2.8620457806076405
Critical Value (10%) : -2.5670392252322554
✅ Reject H0 → Stationary




In [31]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [32]:
results, var_res = granger_var_analysis(df)

 Selected VAR Lag Order: 9

Granger Causality Tests (finbert7, fingpt7 → return)
--------------------------------------------------
Predictor                    F-stat     p-value
--------------------------------------------------
finbert7                      1.307      0.2272
fingpt7                       0.668       0.739
Joint (finbert7, fingpt7)     1.271      0.1955
--------------------------------------------------


# JPM

In [33]:
prices = preparing_OLCHV("JPM")
news = preparing_news("JPM")

df = merge_data(prices, news)

adf_test(df["Close"], title="Close Price")
adf_test(df["return"], title="Log Return")

--- ADF Test: Close Price ---
ADF Statistic : 2.377052626649117
p-value : 0.9989990981516066
# Lags Used : 29
# Observations Used : 5695
Critical Value (1%) : -3.4314987708423077
Critical Value (5%) : -2.86204764612708
Critical Value (10%) : -2.5670402183037195
❌ Fail to Reject H0 → Non-stationary


--- ADF Test: Log Return ---
ADF Statistic : -17.570708812521268
p-value : 4.0528716463236146e-30
# Lags Used : 20
# Observations Used : 5704
Critical Value (1%) : -3.4314969574489025
Critical Value (5%) : -2.86204684493632
Critical Value (10%) : -2.567039791806001
✅ Reject H0 → Stationary




In [34]:
# granger_test(df, "avg_finbert", "return", MX_LAG)
# granger_test(df, "finbert7", "return", MX_LAG)
# granger_test(df, "avg_fingpt", "return", MX_LAG)
# granger_test(df, "fingpt7", "return", MX_LAG)

In [35]:
results, var_res = granger_var_analysis(df)

 Selected VAR Lag Order: 9

Granger Causality Tests (finbert7, fingpt7 → return)
--------------------------------------------------
Predictor                    F-stat     p-value
--------------------------------------------------
finbert7                      1.754     0.07166
fingpt7                       1.723     0.07807
Joint (finbert7, fingpt7)     1.728     0.02807
--------------------------------------------------
